# UNSW-NB15: shared preprocessing workflow

The six stages live in `utils/preprocessing.py`. Dataset-specific semantics live in `utils/dataset_configs.py`. This notebook selects a configuration, runs the common pipeline once, and displays each stage's results. Use the project's `.venv` kernel.

Generator fitting uses development normals only. Boundary vocabularies use all development rows. Validation stays held out. CSV and Parquet/NPZ copies are saved under a separate output directory for each dataset; rerunning overwrites that dataset's artifacts.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = next(
    (p for p in (cwd, *cwd.parents) if (p / "utils" / "preprocessing.py").is_file()), None
)
if project_root is None:
    raise FileNotFoundError("Start the kernel inside the project repository")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.dataset_configs import UNSW_NB15
from utils.preprocessing import plot_distributions, run_preprocessing

config = UNSW_NB15
result = run_preprocessing(
    project_root / "data" / "UNSW-NB15" / "train.csv",
    config,
    output_root=project_root / "output",
    seed=42,
    validation_fraction=0.20,
)
tables = result["audit_tables"]
print("Dataset:", config.name, "Output:", result["output_dir"])

## Step 1: schema and data audit

Inspect semantics, quality, duplicate groups, class-wise numerical distributions, and categorical support. These full-data tables are exploratory and are not used as fitted transformation parameters. Zero is a valid normal value. Category strings such as `-` remain valid values.

In [ ]:
display(result["train_df"].head())
with pl.Config(tbl_rows=90, tbl_cols=16):
    for name in [
        "feature_catalogue",
        "data_quality",
        "duplicate_summary",
        "numerical_distributions",
        "category_frequencies",
    ]:
        print(name)
        display(tables[name])
fig = plot_distributions(result)
plt.show()

## Step 2: development and validation splits

An approximately 80/20 class-stratified split uses seed 42 and keeps exact feature duplicates together. Validation contains normals and attacks; it does not reproduce genuinely unseen attack families. Conflicting labels within a duplicate group stop the pipeline.

In [ ]:
display(tables["split_summary"])
display(result["split_category_support"])
development_df = result["development_df"]
validation_df = result["validation_df"]
development_normal_df = result["development_normal_df"]
development_attack_df = result["development_attack_df"]

## Step 3: fitted preprocessing rules

Normal-development vocabularies, constants, bounds, and transformers are separate from boundary vocabularies. Highly skewed active numerical columns use Gaussian quantile transforms; others use standard scaling. Binary and small discrete states remain discrete. Unknown generator categories are rejected; boundary unknowns map to -1 for the later classifier adapter. Empirical normal bounds are distinct from physical constraints.

In [ ]:
with pl.Config(tbl_rows=50):
    display(tables["preprocessing_rules"])
display(tables["unknown_categories"])
preprocessing_metadata = result["preprocessing_metadata"]

## Step 4: GMM and TabDDPM inputs

GMM receives float64 numericals and aligned whole discrete donor tuples. TabDDPM receives float32 numericals and int64 category codes, including binary/discrete states. Constants are omitted and saved for reconstruction. Source-row IDs are provenance, not model features. Zero indicators remain an optional later experiment.

In [ ]:
display(tables["model_input_summary"])
gmm_X = result["gmm_X"]
gmm_donors = result["gmm_donors"]
tabddpm_X_num = result["tabddpm_X_num"]
tabddpm_X_cat = result["tabddpm_X_cat"]
model_input_metadata = result["model_input_metadata"]
print("Discrete columns:", model_input_metadata["discrete_columns"])
print("Cardinalities:", model_input_metadata["categorical_cardinalities"])

## Step 5: reusable decoding

The shared decoder restores original scales, constants, categories, dtypes, and feature order. GMM generation must supply whole donor tuples selected conditional on sampled components. The example reconstructs fitting data; it does not generate synthetic samples. Decode generated candidates before boundary filtering. Float32 numerical precision can amplify inverse-transform errors; Step 6 measures them using per-row precision bounds.

In [ ]:
from utils import TabularDecoder

decoder = TabularDecoder.from_directory(result["model_input_output"])
decoded_gmm = decoder.decode_gmm(gmm_X, gmm_donors)
decoded_tabddpm = decoder.decode_tabddpm(tabddpm_X_num, tabddpm_X_cat)
display(decoded_tabddpm.head())

## Step 6: final artifact checks and audit export

Checks cover source/split provenance, held-out separation, normal-only fitting, array dimensions/codes, donor alignment, decoding, and CSV copies. Passing establishes artifact consistency, not downstream detection utility. Audit tables are saved as CSV/Parquet, with a JSON report and checksummed inventory.

In [ ]:
print("Audit status:", result["audit_report"]["status"])
display(tables["artifact_checks"].group_by("passed").len())
display(tables["reconstruction_errors"].sort("max_absolute_error", descending=True).head(10))
print("Audit directory:", result["audit_output"])